## **1. Conver Data to Supervised Fine-tuning JSON Format**
---

In [1]:
import os
import json
import random
import pandas as pd
from transformers import set_seed

SEED = 42
set_seed(SEED)

DATA_PATH = '../data/data_model.csv'
DATA_DIR  = '../data/data_sft'
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
SYSTEM_TEMPLATE = (
    'You are a recruitment expert.\n'
    'Task: Estimate the annual salary in USD from the given job and company info.\n'
    'Output rules:\n'
    '- Print one integer only (no text, no commas, no dots, no units).\n'
    '- If uncertain, still output your best single integer.\n'
)

USER_TEMPLATE = (
    'Based on the following information, please estimate the annual salary (USD):\n'
    '- Job Title: {job_simplified}\n'
    '- Seniority: {seniority}\n'
    '- Company: {Company Name}\n'
    '- Location: {Location}\n'
    '- State/Province: {job_state}\n'
    '- Headquarters: {Headquarters}\n'
    '- Company Size: {Size}\n'
    '- Ownership Type: {Type of ownership}\n'
    '- Industry: {Industry}\n'
    '- Sector: {Sector}\n'
    '- Revenue: {Revenue}\n'
    '- Required Skills: Python={Python_yn}, Spark={Spark}, AWS={AWS_yn}\n'
    '\nPlease respond with a SINGLE NUMBER representing the *annual salary* (USD).'
)

COLS_ORDER = [
    'job_simplified', 'seniority', 'Company Name', 'Location', 'job_state', 'Headquarters', 'Size', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Python_yn', 'Spark', 'AWS_yn'
]

In [3]:
def build_example(row):
    return {
        'messages': [
            {'role': 'system'   , 'content': SYSTEM_TEMPLATE},
            {'role': 'user'     , 'content': USER_TEMPLATE.format(**row[COLS_ORDER])},
            {'role': 'assistant', 'content': str(int(round(float(row['Average Salary']))))}
        ]
    }

In [4]:
df  = pd.read_csv(DATA_PATH)
n   = len(df)
idx = list(range(n))
random.shuffle(idx)

n_test = int(n * 0.15)
n_val  = int(n * 0.15)

splits  = {'train': set(idx[n_test+n_val:]), 'val': set(idx[n_test:n_test+n_val]), 'test': set(idx[:n_test])}
holders = {k: [] for k in splits}

for i, row in df.iterrows():
    example = build_example(row)
    for split, idset in splits.items():
        if i in idset:
            holders[split].append(example)
            break

for split, data in holders.items():
    with open(f"{DATA_DIR}/{split}.json", 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
            
print({k: len(v) for k, v in holders.items()})

{'train': 520, 'val': 111, 'test': 111}


## **2. Fine-tuning with QLoRA and SFT**
---

In [5]:
%pip uninstall -q torchvision -y

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import wandb
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv('../.env')

login(os.getenv('HF_TOKEN'))
wandb.login(key=os.getenv('WANDB_API_KEY'))

# Disable multi-GPU training
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# WandB configuration
os.environ['WANDB_PROJECT']   = 'SFT-QLoRA-Llama-3.1-8B-Instruct'
os.environ['WANDB_LOG_MODEL'] = 'false'
os.environ['WANDB_WATCH']     = 'false'

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuitc2502 (YuITC-LLM) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
import torch
from datasets     import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl          import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM, is_conversational
from peft         import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL  = 'meta-llama/Llama-3.1-8B-Instruct'
TUNED_MODEL = 'YuITC/llama31-8b-ins-qlora-sft'
OUTPUT_DIR  = '../outputs/qlora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
# Quantize model    
quantization_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto', quantization_config=quantization_config, torch_dtype='auto')
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)
model = get_peft_model(model, lora_config)
model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

tokenizer.chat_template = """{% for message in messages %}
{% if message['role'] == 'system' %}<|system|>\n{{ message['content'] }}{% endif %}
{% if message['role'] == 'user' %}<|user|>\n{{ message['content'] }}{% endif %}
{% if message['role'] == 'assistant' %}<|assistant|>\n{% generation %}{{ message['content'] }}{% endgeneration %}{% endif %}
{% endfor %}"""

In [10]:
# Load conversational dataset
dataset = load_dataset('json', data_files={
    'train'     : f"{DATA_DIR}/train.json",
    'validation': f"{DATA_DIR}/val.json",
    'test'      : f"{DATA_DIR}/test.json"
})

assert all(is_conversational(example) for example in dataset['train'])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [11]:
# Data collator
data_collator = DataCollatorForCompletionOnlyLM(
    tokenizer            = tokenizer, 
    instruction_template = '<|user|>\n', 
    response_template    = '<|assistant|>\n'
)

/root/2024-DataScience-Salaries-Analysis/.venv/lib/python3.10/site-packages/trl/trainer/utils.py:126: UserWarning: The pad_token_id and eos_token_id values of this tokenizer are identical. If you are planning for multi-turn training, it can result in the model continuously generating questions and answers without eos token. To avoid this, set the pad_token_id to a different value.
  warnings.warn(


In [12]:
# Configure SFT
sft_config = SFTConfig(
    output_dir                  = OUTPUT_DIR, 
    report_to                   = 'wandb',
    run_name                    = '2708-2305-push-to-hf-final',
    seed                        = SEED,
    
    max_length                  = 384,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 16,
    num_train_epochs            = 5,
    
    learning_rate               = 2e-4,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.1,
    assistant_only_loss         = True,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    logging_steps               = 5,
    
    fp16                        = True,
    gradient_checkpointing      = True,
    optim                       = 'adamw_bnb_8bit',
    
    hub_model_id                = TUNED_MODEL,
    hub_private_repo            = False,
    push_to_hub                 = True,
    hub_strategy                = 'end'
)

In [13]:
# Training
trainer = SFTTrainer(
    model            = model,
    args             = sft_config, 
    train_dataset    = dataset['train'],
    eval_dataset     = dataset['validation'],
    processing_class = tokenizer,
    data_collator    = data_collator,
)

print("Starting training...")
trainer.train()

Tokenizing train dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/111 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/111 [00:00<?, ? examples/s]

Starting training...


Epoch,Training Loss,Validation Loss
1,2.904300,2.765724
2,2.546300,2.591776
3,2.389100,2.428019
4,1.735700,2.187189
5,1.049400,2.224821


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


TrainOutput(global_step=165, training_loss=2.3634538332621258, metrics={'train_runtime': 2652.4233, 'train_samples_per_second': 0.98, 'train_steps_per_second': 0.062, 'total_flos': 2.384632889708544e+16, 'train_loss': 2.3634538332621258})

In [14]:
# Save model
print("Saving model...")
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

Saving model...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...alysis/outputs/qlora/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...uts/qlora/adapter_model.safetensors:   1%|          |  558kB / 83.9MB            

  ...sis/outputs/qlora/training_args.bin:   3%|3         |   191B / 5.82kB            

('../outputs/qlora/tokenizer_config.json',
 '../outputs/qlora/special_tokens_map.json',
 '../outputs/qlora/chat_template.jinja',
 '../outputs/qlora/tokenizer.json')

In [15]:
# Finish wandb log
wandb.finish()

eval/loss,█▆▄▁▁
eval/mean_token_accuracy,▁▂▄▆█
eval/num_tokens,▁▃▅▆█
eval/runtime,▄▄▁▁█
eval/samples_per_second,▅▅██▁
eval/steps_per_second,▅▅██▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▁▂▂▂▂▂▂
train/learning_rate,▂▄▆██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,█▄▃▃▃▃▃▃▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁


## **3. Evaluate Fine-tuned Model**
---

In [16]:
import re
import numpy as np
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft         import AutoPeftModelForCausalLM

HF_MODEL_ID = TUNED_MODEL

In [17]:
# # Load tokenizer
tuned_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, use_fast=True)
if tuned_tokenizer.pad_token is None:
    tuned_tokenizer.pad_token = tuned_tokenizer.eos_token
tuned_tokenizer.padding_side = 'right'

In [18]:
# Load fine-tuned model
quantization_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = torch.float16,
)

tuned_model = AutoPeftModelForCausalLM.from_pretrained(HF_MODEL_ID, device_map='auto', torch_dtype=torch.float16, quantization_config=quantization_config)
tuned_model.eval()
tuned_model.config.use_cache = True

adapter_config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

In [19]:
# Build prompt for inference
def build_prompt_for_inference(messages, tokenizer):
    msgs = [m for m in messages if m['role'] != 'assistant']
    msgs.append({'role': 'assistant', 'content': ''})
    return tokenizer.apply_chat_template(msgs, tokenize=False)

In [20]:
# Get mean of train set
y_train = []
for example in dataset['train']:
    try:
        y_train.append(int(str(example['messages'][-1]['content']).strip()))
    except:
        print(f"Error for example: {example}")
        pass

train_mean = int(round(float(np.mean(y_train)))) if len(y_train) else None
train_mean

102601

In [21]:
# Inference on test set
def predict_one(prompt, max_new_tokens=9, int_pattern=re.compile(r"-?\d+")):
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = False,
            eos_token_id       = tokenizer.eos_token_id,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.0,
        )
        
    gen       = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    gen_clean = gen.replace(',', '')
    
    m = int_pattern.search(gen_clean)
    
    return (int(m.group(0)) if m else None), gen


preds, gts, raw_texts = [], [], []

for example in tqdm(dataset['test'], desc='Predicting'):
    messages = example['messages']
    try:
        y_true = int(str(messages[-1]['content']).strip())
    except:
        print(f"Error for example: {example}")
        continue
    gts.append(y_true)

    prompt      = build_prompt_for_inference(messages, tokenizer)
    y_pred, raw = predict_one(prompt)
    
    if y_pred is None:
        print(f"Warning: No integer found in model output. Using train mean {train_mean}.")
        y_pred = train_mean if train_mean is not None else 0
        
    preds.append(int(y_pred))
    raw_texts.append(raw)

Predicting: 100%|██████████| 111/111 [02:08<00:00,  1.16s/it]


In [22]:
print(f"RMSE: {root_mean_squared_error(gts, preds):.3f}")
print(f"MAE : {mean_absolute_error(gts, preds):.3f}")
print(f"R2  : {r2_score(gts, preds):.3f}")

RMSE: 352336.183
MAE : 101595.676
R2  : -94.884
